In [ ]:
import numpy as np
from firedrake import *
from firedrake.cython import dmcommon
from petsc4py import PETSc
import math

In [ ]:
# parameters in SI units
t_end = 5.0  # time of simulation [s]
dt = 0.005  # time step [s]
g = 9.8  # gravitational acceleration
# water
Lx = 20.0  # length of the tank [m] in x-direction; needed for computing initial condition
Lz = 10.0  # height of the tank [m]; needed for computing initial condition
rho = 1000.0  # fluid density in kg/m^2 in 2D [water]
# solid parameters
#  - we use a sufficiently soft material to be able to see noticeable structural displacement
rho_B = 7700.0  # structure density in kg/m^2 in 2D
lam = 1e7  # N/m in 2D - first Lame constant
mu = 1e7  # N/m in 2D - second Lame constant

# these numbers must match the ones defined in the mesh file
fluid_id = 1  # fluid subdomain
structure_id = 2  # structure subdomain
bottom_id = 1  # structure bottom
top_id = 6  # fluid surface
interface_id = 9  # fluid-structure interface

L = Lz
T = L / math.sqrt(g * L)
t_end /= T
dt /= T
Lx /= L
Lz /= L
rho_B /= rho
lam /= g * rho * L
mu /= g * rho * L
rho = 1.0  # or equivalently rho /= rho

mesh = Mesh("/home/charlottecai/MSc_Project_new/L_domain.msh")
dim = 2

# Extract submesh
mesh_F = Submesh(mesh, dim, fluid_id,label_name=dmcommon.CELL_SETS_LABEL, name="fluid_mesh")
x_F = SpatialCoordinate(mesh_F)
n_F = FacetNormal(mesh_F)

mesh_S = Submesh(mesh, dim, structure_id, label_name=dmcommon.CELL_SETS_LABEL, name="solid_mesh")
x_S = SpatialCoordinate(mesh_S)
n_S = FacetNormal(mesh_S)

# Function spaces
V_W = FunctionSpace(mesh_F, "CG", 1)
V_B = VectorFunctionSpace(mesh_S, "CG", 1)

mixed_V = V_W * V_B

# Measures
dx_F = Measure("dx", mesh_F)
dx_S = Measure("dx", mesh_S)

# interface measures
ds_F = Measure("ds", mesh_F, intersect_measures=(Measure("ds", mesh_S),))
ds_S = Measure("ds", mesh_S, intersect_measures=(Measure("ds", mesh_F),))

# fluid domain
phi = Function(V_W, name="phi")
phi_f = Function(V_W, name="phi_f")
eta = Function(V_W, name="eta")

trial_W = TrialFunction(V_W)
v_W = TestFunction(V_W)

# Structure domain
X = Function(V_B, name="X")
U = Function(V_B, name="U")

trial_B = TrialFunction(V_B)
v_B = TestFunction(V_B)

# mixed domain
trial_f, trial_s = TrialFunctions(mixed_V)
v_f, v_s = TestFunctions(mixed_V)

tmp_f = Function(V_W)
tmp_s = Function(V_B)
result_mixed = Function(mixed_V)

# Boundary conditions
BC_bottom = DirichletBC(V_B, as_vector([0.0, 0.0]), bottom_id)
BC_bottom_mixed = DirichletBC(mixed_V.sub(1), as_vector([0.0, 0.0]), bottom_id)
BC_phi_f = DirichletBC(mixed_V.sub(0), phi_f, top_id)

# Consider free surface
class MyBC(DirichletBC):
    def __init__(self, V, value, markers):
        super(MyBC, self).__init__(V, value, 0)
        self.nodes = np.unique(np.where(markers.dat.data_ro_with_halos == 0)[0])

def surface_BC():
    bc = DirichletBC(V_W, 1, top_id)
    f = Function(V_W, dtype=np.int32)
    bc.apply(f)
    return MyBC(V_W, 0, f)

BC_exclude_beyond_surface = surface_BC()

# solvers
# 1. equations for phi at the free surface
a_phi_f = trial_W * v_W * ds_F(top_id)
L_phi_f = (phi_f - dt * eta) * v_W * ds_F(top_id)
LVP_phi_f = LinearVariationalProblem(a_phi_f, L_phi_f, phi_f, bcs=BC_exclude_beyond_surface)
LVS_phi_f = LinearVariationalSolver(LVP_phi_f)

# 2. equations for  X (beam displacement)
a_X = dot(trial_B, v_B) * dx_S
L_X = dot((X + dt * U), v_B) * dx_S

LVP_X = LinearVariationalProblem(a_X, L_X, X, bcs=[BC_bottom])
LVS_X = LinearVariationalSolver(LVP_X)

# 3. equations for phi at the fluid domain, U and eta
delX = nabla_grad(X)
delv_B = nabla_grad(v_s)
T_x_dv = lam * div(X) * div(v_s) + mu * inner(delX, delv_B + transpose(delv_B))
a_U = rho_B * dot(trial_s, v_s) * dx_S
L_U = (rho_B * dot(U, v_s) - dt * T_x_dv) * dx_S
a_phi = dot(grad(trial_f), grad(v_f)) * dx_F

#a_U += dot(v_s, n_S) * trial_f * ds_S(interface_id)
#L_U += dot(v_s, n_S) * phi * ds_S(interface_id)
#a_phi += -dot(n_F, trial_s) * v_f * ds_F(interface_id)

a_U += dot(v_s, n_F) * trial_f * ds_F(interface_id)
L_U += dot(v_s, n_F) * phi * ds_F(interface_id)
a_phi += -dot(n_F, trial_s) * v_f * ds_F(interface_id)

LVP_U_phi = LinearVariationalProblem(
    a_U + a_phi,
    L_U,
    result_mixed,
    bcs=[BC_phi_f, BC_bottom_mixed]
)

LVS_U_phi = LinearVariationalSolver(LVP_U_phi)

# eta
a_eta = trial_W * v_W * ds_F(top_id)
L_eta = (eta * v_W * ds_F(top_id) + dt * dot(grad(v_W), grad(phi)) * dx_F)
L_eta += - dt * dot(n_F, U) * v_W * ds_F(interface_id) ###

LVP_eta = LinearVariationalProblem(a_eta, L_eta, eta, bcs=BC_exclude_beyond_surface)
LVS_eta = LinearVariationalSolver(LVP_eta)

In [69]:
# initial condition
n_mode = 1
a = 0.0 * T / L ** 2
b = 5.0 * T / L ** 2
lambda_x = np.pi * n_mode / Lx
omega = np.sqrt(lambda_x * np.tanh(lambda_x * Lz))
phi_exact_expr = a * cos(lambda_x * x_F[0]) * cosh(lambda_x * x_F[1])
eta_exact_expr = -omega * b * cos(lambda_x * x_F[0]) * cosh(lambda_x * Lz)

bc_top = DirichletBC(V_W, 0, top_id)
eta.assign(0.0)
phi.assign(0.0)
eta_exact = Function(V_W)
eta_exact.interpolate(eta_exact_expr)
eta.assign(eta_exact, bc_top.node_set)
phi.interpolate(phi_exact_expr)
phi_f.assign(phi, bc_top.node_set)

Coefficient(WithGeometry(FunctionSpace(<firedrake.mesh.MeshTopology object at 0x79312a591720>, FiniteElement('Lagrange', triangle, 1), name=None), Mesh(VectorElement(FiniteElement('Lagrange', triangle, 1), dim=2), 634)), 1244)

In [71]:
for step, t in enumerate(np.linspace(0, t_end, int(t_end / dt))):
    LVS_phi_f.solve()
    LVS_U_phi.solve()

    tmp_f, tmp_s = result_mixed.subfunctions
    phi.assign(tmp_f)
    U.assign(tmp_s)

    LVS_eta.solve()
    LVS_X.solve()

    if step % 20 == 0:
        phi_norm = sqrt(abs(assemble(dot(phi, phi) * dx_F)))
        eta_norm = sqrt(abs(assemble(dot(eta,eta) * ds_F(top_id))))
        U_norm = sqrt(abs(assemble(dot(U, U) * dx_S)))
        X_norm = sqrt(abs(assemble(dot(X, X) * dx_S)))
        
        PETSc.Sys.Print(
        "step =", step,
        "t =", float(t),
        "phi_norm =", float(phi_norm),
        "eta_norm =", float(eta_norm),
        "U_norm =", float(U_norm),
        "X_norm =", float(X_norm),
    )

step = 0 t = 0.0 phi_norm = 0.01856494579103592 eta_norm = 0.14121688030724427 U_norm = 0.003912421073232032 X_norm = 0.009828612504016595
step = 20 t = 0.09909404340952618 phi_norm = 0.009734287204815081 eta_norm = 0.14423292537691396 U_norm = 0.0024280319260850287 X_norm = 0.010099318635839198
step = 40 t = 0.19818808681905237 phi_norm = 0.0013534069753355746 eta_norm = 0.14527532803041698 U_norm = 0.0021312921517562234 X_norm = 0.010221177869125622
step = 60 t = 0.2972821302285786 phi_norm = 0.0087773586776425 eta_norm = 0.14436021532388657 U_norm = 0.002996442239996761 X_norm = 0.01019653091443638
step = 80 t = 0.39637617363810473 phi_norm = 0.018088334690871174 eta_norm = 0.14152369834873846 U_norm = 0.004349407701976783 X_norm = 0.010036268081694381
step = 100 t = 0.49547021704763095 phi_norm = 0.027111788038739405 eta_norm = 0.13680625281946443 U_norm = 0.005764191214755485 X_norm = 0.009760608912644323
step = 120 t = 0.5945642604571572 phi_norm = 0.03536018859429333 eta_norm = 